# 验证转换后的权重和标准Group-wise权重对比

这个notebook用于验证转换后的权重和标准的group-wise权重是否一致。


In [1]:
import torch
import numpy as np
from pathlib import Path


In [5]:
# 设置文件路径
converted_weight_path = "/data/zihan/dev_block/scale_groupwise.pt"
standard_weight_path = "/FirstIntelligence/home/zihanm/deepcompressor/examples/diffusion/checkpoints/nvfp4/scale.pt"  # 请填入标准group-wise权重的路径

# 检查文件是否存在
if not Path(converted_weight_path).exists():
    print(f"错误: 转换后的权重文件不存在: {converted_weight_path}")
else:
    print(f"✓ 转换后的权重文件存在: {converted_weight_path}")

if standard_weight_path and not Path(standard_weight_path).exists():
    print(f"错误: 标准权重文件不存在: {standard_weight_path}")
elif standard_weight_path:
    print(f"✓ 标准权重文件存在: {standard_weight_path}")
else:
    print("⚠️  请设置标准权重文件路径")


✓ 转换后的权重文件存在: /data/zihan/dev_block/scale_groupwise.pt
✓ 标准权重文件存在: /FirstIntelligence/home/zihanm/deepcompressor/examples/diffusion/checkpoints/nvfp4/scale.pt


In [6]:
def compare_weight_shapes(converted_dict, standard_dict):
    """
    比较转换后的权重和标准group-wise权重的形状
    
    Args:
        converted_dict: 转换后的权重字典
        standard_dict: 标准的group-wise权重字典
    
    Returns:
        dict: 比较结果
    """
    results = {
        'total_keys': 0,
        'matched_keys': 0,
        'shape_mismatches': [],
        'missing_keys': [],
        'extra_keys': [],
        'successful_matches': []
    }
    
    print("=== 权重形状对比验证 ===")
    print(f"转换后权重键数量: {len(converted_dict)}")
    print(f"标准权重键数量: {len(standard_dict)}")
    print()
    
    # 获取所有键
    converted_keys = set(converted_dict.keys())
    standard_keys = set(standard_dict.keys())
    
    # 检查缺失的键
    missing_in_converted = standard_keys - converted_keys
    missing_in_standard = converted_keys - standard_keys
    
    if missing_in_converted:
        print(f"转换后权重中缺失的键 ({len(missing_in_converted)}个):")
        for key in sorted(missing_in_converted):
            print(f"  - {key}")
        print()
    
    if missing_in_standard:
        print(f"标准权重中缺失的键 ({len(missing_in_standard)}个):")
        for key in sorted(missing_in_standard):
            print(f"  - {key}")
        print()
    
    # 比较共同的键
    common_keys = converted_keys & standard_keys
    results['total_keys'] = len(common_keys)
    
    print(f"共同键数量: {len(common_keys)}")
    print()
    
    for key in sorted(common_keys):
        converted_value = converted_dict[key]
        standard_value = standard_dict[key]
        
        print(f"验证 {key}:")
        
        # 检查类型
        if not isinstance(converted_value, torch.Tensor) or not isinstance(standard_value, torch.Tensor):
            if type(converted_value) == type(standard_value):
                print(f"  ✓ 类型匹配: {type(converted_value)}")
                results['matched_keys'] += 1
                results['successful_matches'].append(key)
            else:
                print(f"  ✗ 类型不匹配: {type(converted_value)} vs {type(standard_value)}")
            continue
        
        # 检查形状
        if converted_value.shape == standard_value.shape:
            print(f"  ✓ 形状匹配: {converted_value.shape}")
            results['matched_keys'] += 1
            results['successful_matches'].append(key)
        else:
            print(f"  ✗ 形状不匹配: {converted_value.shape} vs {standard_value.shape}")
            results['shape_mismatches'].append({
                'key': key,
                'converted_shape': converted_value.shape,
                'standard_shape': standard_value.shape
            })
        
        print()
    
    return results


In [7]:
def print_summary(results):
    """打印比较结果摘要"""
    print("=" * 50)
    print("形状比较结果摘要")
    print("=" * 50)
    
    total = results['total_keys']
    matched = results['matched_keys']
    
    print(f"总键数: {total}")
    print(f"形状匹配键数: {matched}")
    print(f"形状匹配率: {matched/total*100:.2f}%" if total > 0 else "形状匹配率: N/A")
    print()
    
    if results['shape_mismatches']:
        print(f"形状不匹配 ({len(results['shape_mismatches'])}个):")
        for mismatch in results['shape_mismatches']:
            print(f"  {mismatch['key']}: {mismatch['converted_shape']} vs {mismatch['standard_shape']}")
        print()
    
    if results['successful_matches']:
        print(f"形状匹配的键 ({len(results['successful_matches'])}个):")
        for key in results['successful_matches'][:10]:  # 只显示前10个
            print(f"  ✓ {key}")
        if len(results['successful_matches']) > 10:
            print(f"  ... 还有 {len(results['successful_matches']) - 10} 个")


In [8]:
# 加载权重文件并进行形状比较
if standard_weight_path and Path(standard_weight_path).exists():
    print("加载权重文件...")
    
    # 加载转换后的权重
    converted_dict = torch.load(converted_weight_path, map_location="cpu")
    print(f"✓ 转换后权重加载完成，包含 {len(converted_dict)} 个键")
    
    # 加载标准权重
    standard_dict = torch.load(standard_weight_path, map_location="cpu")
    print(f"✓ 标准权重加载完成，包含 {len(standard_dict)} 个键")
    
    # 执行形状比较
    results = compare_weight_shapes(converted_dict, standard_dict)
    
    # 打印摘要
    print_summary(results)
    
    # 保存结果到变量供进一步分析
    comparison_results = results
    
else:
    print("请先设置标准权重文件路径")


加载权重文件...


✓ 转换后权重加载完成，包含 1520 个键
✓ 标准权重加载完成，包含 1520 个键
=== 权重形状对比验证 ===
转换后权重键数量: 1520
标准权重键数量: 1520

共同键数量: 1520

验证 single_transformer_blocks.0.attn.to_k.weight.scale.0:
  ✓ 形状匹配: torch.Size([1, 1, 1, 1])

验证 single_transformer_blocks.0.attn.to_k.weight.scale.1:
  ✓ 形状匹配: torch.Size([3072, 1, 192, 1])

验证 single_transformer_blocks.0.attn.to_k.weight.zero:
  ✓ 形状匹配: torch.Size([])

验证 single_transformer_blocks.0.attn.to_q.weight.scale.0:
  ✓ 形状匹配: torch.Size([1, 1, 1, 1])

验证 single_transformer_blocks.0.attn.to_q.weight.scale.1:
  ✓ 形状匹配: torch.Size([3072, 1, 192, 1])

验证 single_transformer_blocks.0.attn.to_q.weight.zero:
  ✓ 形状匹配: torch.Size([])

验证 single_transformer_blocks.0.attn.to_v.weight.scale.0:
  ✓ 形状匹配: torch.Size([1, 1, 1, 1])

验证 single_transformer_blocks.0.attn.to_v.weight.scale.1:
  ✓ 形状匹配: torch.Size([3072, 1, 192, 1])

验证 single_transformer_blocks.0.attn.to_v.weight.zero:
  ✓ 形状匹配: torch.Size([])

验证 single_transformer_blocks.0.norm.linear.weight.scale.0:
  ✓ 形状匹配: torch.Size([9

In [9]:
# 可选：详细分析形状不匹配的键
if 'comparison_results' in locals() and comparison_results['shape_mismatches']:
    print("=== 详细分析形状不匹配 ===")
    for mismatch in comparison_results['shape_mismatches'][:]:  # 显示前10个
        key = mismatch['key']
        print(f"\n键: {key}")
        print(f"转换后形状: {mismatch['converted_shape']}")
        print(f"标准形状: {mismatch['standard_shape']}")
        
        # 分析形状差异
        converted_shape = mismatch['converted_shape']
        standard_shape = mismatch['standard_shape']
        
        if len(converted_shape) == len(standard_shape):
            print("形状维度相同，差异在:")
            for i, (c, s) in enumerate(zip(converted_shape, standard_shape)):
                if c != s:
                    print(f"  第{i}维: {c} vs {s} (差异: {c-s})")
        else:
            print(f"维度数不同: {len(converted_shape)} vs {len(standard_shape)}")
else:
    print("所有键的形状都匹配！")


所有键的形状都匹配！
